# 04 — Scale-aware precipitation decomposition

Replaces bulk RMSE/correlation framing with a physically interpretable
object-based decomposition. Each precipitating *system* (connected
component above a rain threshold) is described by

$$P = \rho \times A,\qquad \ln P = \ln A + \ln \rho$$

- **A**  precipitating area (km², latitude-weighted)
- **ρ**  area-weighted mean intensity (mm) over that area
- **P**  total precip (mm·km²) carried by the system
- **L = √A**  characteristic system scale (km)

Truth and predicted systems are matched by spatial overlap (greedy max-IoU);
unmatched truth = *miss*, unmatched pred = *false alarm*. Model error is
split log-additively, `dlnP = dlnA + dlnρ` (no cross term), and everything
is stratified by system scale so we can see *which sizes* the model fails on
and whether the failure is one of area or of intensity.

Built on the registry — pick any run; the headless equivalents are
`scripts/eval_scale_decomposition.py` and `scripts/plot_scale_decomposition.py`.

In [ ]:
import sys, pathlib
REPO = pathlib.Path('..').resolve()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import numpy as np
import matplotlib.pyplot as plt

from latentsky import list_runs, load_run
from latentsky.inference import predict_piece
from latentsky.scale_analysis import (
    system_table, summarize_by_scale, default_scale_edges)

list_runs()

## Build the per-system table for one run

`system_table` labels every frame, matches truth↔pred systems and returns
one tidy row per system (role ∈ match/miss/false_alarm).

In [ ]:
RUN_NAME = 'mseuv_lead3h'
PIECE = 32
THRESHOLD = 0.1   # mm — a precipitating pixel
IOU_MIN = 0.1     # min overlap for a truth<->pred match

run = load_run(RUN_NAME)
_, Y, Y_pred = predict_piece(run, PIECE, batch_size=4)
df = system_table(Y, Y_pred, threshold=THRESHOLD, iou_min=IOU_MIN)
print(df['role'].value_counts())
df.head()

Sanity checks — the two identities must hold exactly for matched systems:

In [ ]:
m = df[df.role == 'match']
assert np.allclose(m.dlnA + m.dlnrho, m.dlnP)
assert np.allclose(m.rho_t * m.A_t, m.P_t)
assert np.allclose(m.rho_p * m.A_p, m.P_p)
print('P = rho*A and dlnP = dlnA + dlnrho verified')

## Scale binning + summary

`summarize_by_scale` returns per-scale-bin object skill (POD/FAR), the
median area/intensity log-error, the precip-bias budget, and the
unpaired truth-vs-pred distributions of A, ρ, P.

In [ ]:
edges = default_scale_edges()        # log-spaced L bins, km
ss = summarize_by_scale(df, edges)
ss.by_scale[['scale_lo','scale_hi','n_match','n_miss','n_fa',
             'POD','FAR','dlnA_med','dlnrho_med','area_frac','int_frac',
             'miss_P_frac','fa_P_frac']].round(3)

## System-size distribution — truth vs predicted

In [ ]:
Lt = df.loc[df.L_t.notna(), 'L_t']
Lp = df.loc[df.L_p.notna(), 'L_p']
be = np.geomspace(min(Lt.min(), Lp.min()), max(Lt.max(), Lp.max()), 30)
plt.hist(Lt, bins=be, histtype='step', color='k', lw=1.8, label='truth')
plt.hist(Lp, bins=be, histtype='step', color='C3', lw=1.8, label='pred')
plt.xscale('log'); plt.yscale('log')
plt.xlabel('system scale L = √A  (km)'); plt.ylabel('# systems')
plt.legend(); plt.grid(True, which='both', alpha=0.25)
plt.title(f'{RUN_NAME} — system-size distribution'); plt.show()

## Area vs intensity error by scale

Which is the model wrong about, and at what sizes? Below zero = under-prediction.

In [ ]:
g = ss.by_scale.sort_values('scale_mid'); x = g.scale_mid
plt.axhline(0, color='k', lw=0.8, ls='--', alpha=0.5)
plt.plot(x, g.dlnA_med, 'o-', color='C0', label='dln A (area)')
plt.plot(x, g.dlnrho_med, 's-', color='C3', label='dln ρ (intensity)')
plt.plot(x, g.dlnP_med, '^:', color='0.4', label='dln P (total)')
plt.xscale('log'); plt.xlabel('system scale (km)')
plt.ylabel('median log error (pred/truth)')
plt.legend(); plt.grid(True, which='both', alpha=0.25)
plt.title(f'{RUN_NAME} — area vs intensity error by scale'); plt.show()

## Full six-panel figure (headless script)

For the canonical figure across runs, run the headless scripts under PBS:

```bash
qsub submit/eval_scale.pbs     # -> docs/scale_systems.parquet + docs/scale_summary.csv
qsub submit/plot_scale.pbs     # -> docs/figures/scale_decomp_<run>.png
```

`scripts/plot_scale_decomposition.py --run <name>` reproduces the six-panel
view (size distribution, precip-by-scale, detection vs scale, area/intensity
error, bias budget, intensity PDF) for any run.